<font color=#009afa size=50>DAT-6002 - Artificial Intelligence</font><br>
<font size=50>WRIT1 Part 1 - Semantic Networks</font><br>
<font size=20 color=#00c382>Medieval Weapons</font><br>

|**Assessment Code:** |WRIT1|
|---|---|
|**Author:**|Tom Rowan|
|**Course:**|DAT6002_S2_25|
|**Course Tutor:**|Imtiaz Hussain Khan, PhD|


**Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.**


---

# Initialise Notebook

In [87]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [88]:
# Standard Libraries
import sys
import time
import json
import random
from datetime import datetime
import re
import csv

# Requests
import requests

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
%matplotlib inline

# Colorama
from colorama import Fore, Back, Style

# NetworkX
import networkx as nx

# PyVis
from pyvis.network import Network

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Create a Timestamp

In [89]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
now_date_timestamp = str(today.strftime("%Y%m%d"))
now_datetime_timestamp = str(today.strftime("%Y%m%d-%H%M"))

### Functions for Pretty Output

print_output - Print coloured Output

In [90]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

### Check for Internet Access

In [91]:
_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.github.com',
                  'https://datascience.daemonchild.com']

Set this to False if you have a local dataset

In [92]:
internet_required = False

In [93]:
# Function to Safely Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online

# Perform the checks
if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    online = False


# Stop running if Internet is required
if online == False & internet_required == True:
    sys.exit('Stopping notebook run here.')    


[INTERNET] Internet Connectivity Detected. Using Internet resources.


---
# Define Semantic Network Class

This class defines a semantic network for any data. The primary purpose is to allow simple gathering of data and to then visualise the network.

In [94]:
# Define Class SemanticNetwork
class SemanticNetwork:

    # Constructor
    def __init__(self, root_node="RootNode",title = "A Semantic Network", **kwargs):

        # Create graph object
        self.G = nx.DiGraph()
        self.root = root_node
        
        # Descriptive title for the network
        self.title = title

        # Handle Additional Arguments
        palette =   {'bg':"#000000", 
                     'text': "#ffffff", 
                     'edge_label': "#F0D90A", 
                     'edge_struct': "#c4830b81", 
                     'edge_prop': "#7847ff81",
                     0:"#C3ADFF", 
                     1:"#9670ff", 
                     2:"#7847ff", 
                     3:"#5A1FFF", 
                     4:"#4100F5", 
                     5:"#3100B8", 
                     6:"#22007C"}

        # Define palette (from https://coolors.co, using shades option)
        #self.bgcolour = palette['bg']
        #self.node_text_colour = palette['text']
        #self.edge_label_colour = palette['edge_label']
        #self.edge_colour_structural = palette['edge_struct']
        #self.edge_colour_property = palette['edge_prop']
        self.palette = palette


    # Add structural relationships to the network (IS-A type)
    def add_relation(self, node, parent):
        self.G.add_edge(node, parent, label="is_a")


    # Add property relationships to the network (HAS-PART type)
    def add_property(self, node, label, value):
        self.G.add_edge(node, value, label=label)
    

    # Calculates distance from the root node
    # Used for dynamic scaling and chosing colours from palette
    def get_node_level(self, node):
        try:
            # Only use 'is_a' edges
            # This prevents property nodes from making shorter paths!
            structural_edges = [(u, v) for u, v, d in self.G.edges(data=True) if d.get('label') == 'is_a']
            structural_subgraph = nx.DiGraph(structural_edges)
            
            # Calculate distance to root using only the calculated graph
            return nx.shortest_path_length(structural_subgraph, source=node, target=self.root)
        except Exception:
            # If the root is unreachable
            return None    


    # Export Network to json file 
    def export_to_json(self, filename):
        #data = nx.edges(self.G)
        data = nx.node_link_data(self.G)    # NB deprecated, but 'edges' fails...

        try: 
            with open(filename, 'w') as f:
                json.dump(data, f, indent=4)

            print_output(f"Successfully saved", f"{len(self.G.nodes)} nodes to {filename}")
        except Exception:
            print_output(f"Failed",f"saving to {filename}", colour='red')


    # Import Network from json file 
    def import_from_json(self, filename):
        
        try:
            with open(filename, 'r') as f:
                data = json.load(f)

            # Destructive
            self.G = nx.node_link_graph(data)
            print_output(f"Successfully loaded", f"{len(self.G.nodes)} nodes from {filename}")
        except Exception:
            print_output(f"Failed",f"saving to {filename}", colour='red')


    
    def get_parent_from_fact(self, fact_string):
        # This splits the string at the comma, takes the second half, 
        # then strips spaces and the ').'
        # Example: "is_a( springald, stone_thrower )." -> "stone_thrower"
        try:
            return fact_string.split(',')[1].split(')')[0].strip()
        except IndexError:
            return fact_string # Fallback if string is malformed



    # The network can export itself to Prolog
    # Requires a rules file to be appended to the output
    def export_to_prolog (self, filename="prolog_file.pl",rules_filename='rules.pl', sort=True):

        # Separate storage for different types of facts
        is_a_facts = []
        has_a_facts = []
        # Empty File
        prolog_file = []

        for source, target, data in self.G.edges(data=True):
            label = data.get('label')

            src = f"{source.lower().replace(' ','_')}"
            tgt = f"{target.lower().replace(' ','_')}"
            label = label.lower().replace('-','_')
                
            # Grouping types together
            if label == 'is_a':

                level = self.get_node_level(target)
                if level == None:
                    level = 99
                    print (src,target)
                if src == self.root: level = 0
                is_a_facts.append((level, f'is_a( {src}, {tgt} ).'))

            else:
                # Special case when there is a 'from x to y' value (eg: 2,3 --> (2,3))
                if ',' in tgt:
                    #tgt = f"{tgt.replace('-','_').replace('_',',')}"
                    has_a_facts.append(f"has_a( {src}, {label},({tgt}) ).")    
                else:
                    has_a_facts.append(f"has_a( {src}, {label},{tgt} ).")

        # Sort the facts
        if sort:
            # Sorts by the first element of the tuple (level)
            is_a_facts.sort(key=lambda x: (x[0], self.get_parent_from_fact(x[1])))
            has_a_facts.sort()

        # Clean the tuples back into strings for output
        is_a_output = [item[1] for item in is_a_facts]    

        # Construct the file content in blocks
        
        prolog_file.append(f"% {self.title} in Prolog")
        prolog_file.append(f"% Auto generated from SemanticNetwork Class, by Tom Rowan")    
        
        prolog_file.append("\n% is_a facts")
        prolog_file.append(f"\n% root_node is {self.root}\n")
        prolog_file.extend(is_a_output)
        
        prolog_file.append("\n% has_a facts\n")
        prolog_file.extend(has_a_facts)

        # Append the rules
        with open(rules_filename, 'r') as f:
            rules_data = f.read()

        prolog_file.append(rules_data)

        # Save to file
        with open(filename, 'w') as f:
            for line in prolog_file:
                f.write(str(line)+'\n')
        
        print_output("Wrote File",f"Prolog written to {filename}.")


    

    # The network can be imported from a prolog file provided it follows the right structure
    #
    #   File must be formatted as follows:
    #       is_a(mammal, animal)
    #       is_a(canine, mammal)
    #       ..
    #       has_a(canine, makes_noise(bark))
    #       has_a(canine, makes_noise(growl))
    #       has_a(canine, legs(4))
    #
    def import_from_prolog (self, filename='prolog_file.pl'):

        # Empty lists
        is_a_list = []
        has_a_list = []

        # Regex patterns
        is_a_pattern = r"^is_a\(\s*(\w+),\s*(\w+)\s*\)\."
        has_a_pattern = r"^has_a\(\s*(\w+),\s*(\w+)\((.*?)\)\s*\)\."

        with open(filename, 'r') as file:
            data = file.read()

            # Extracting is_a facts
            is_a_list = re.findall(is_a_pattern, data, re.MULTILINE)
            
            # Extracting has_a facts
            has_a_list = re.findall(has_a_pattern, data, re.MULTILINE)

        # Add elements to the network
        for fact in is_a_list:
            entity = fact[0]
            parent = fact[1]
            self.add_relation(entity, parent)
    
        for fact in has_a_list:
            entity = fact[0]
            property = fact[1]
            value = fact[2]
            self.add_property(entity, property, value)

        print_output ("Imported network from prolog file", filename)
        print(f" - Added {str(len(is_a_list))} IS_A facts")
        print(f" - Added {str(len(has_a_list))} HAS_A facts")


    # Export the network to a CSV file
    def export_to_csv(self, filename):

        # Identify all unique property labels for headers
        dynamic_headers = set()
        for _, _, data in self.G.edges(data=True):
            label = data.get('label')
            if label and label != 'is_a':
                dynamic_headers.add(label)
        
        sorted_headers = ['node_name', 'parent_category'] + sorted(list(dynamic_headers))

        # Identify "Entity" nodes
        # We only want is_a nodes
        entity_nodes = []
        for node in self.G.nodes():
            # Check if this node is a child or a parent in an is_a relationship
            is_entity = any(d.get('label') == 'is_a' for _, _, d in self.G.in_edges(node, data=True)) or \
                        any(d.get('label') == 'is_a' for _, _, d in self.G.out_edges(node, data=True))
            
            if is_entity:
                entity_nodes.append(node)

        rows = []

        # Build rows only for Entity nodes
        for node in entity_nodes:
            row = {'node_name': node}
            
            # Get parent category
            parents = [target for _, target, data in self.G.out_edges(node, data=True) if data.get('label') == 'is_a']
            row['parent_category'] = parents[0] if parents else "Root"

            # Collect properties for the node
            for _, target, data in self.G.out_edges(node, data=True):
                label = data.get('label')
                if label != 'is_a':
                    # Join multiple values (e.g., used_by: knight; commoners)
                    if row.get(label):
                        row[label] = f"{row[label]}; {target}"
                    else:
                        row[label] = target

            rows.append(row)

        # 4. Write to File
        try:
            with open(filename, 'w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=sorted_headers)
                writer.writeheader()
                writer.writerows(rows)

            print_output(f"Wrote File",f"Exported {len(rows)} entity rows to {filename}")

        except Exception as e:
            print_output("Failed", f"An error occurred: {e}", colour='red')


        # Import Network from CSV file
    def import_from_csv(self, filename):
        import csv
        
        # Keep track of counts for reporting
        is_a_count = 0
        has_a_count = 0
        
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                
                for row in reader:
                    # Get the node entity name
                    node = row.get('node_name')

                    if not node: 
                        continue # Skip empty rows
                        
                    # Clean up node string
                    node = node.strip()

                    if node == self.root:
                        continue
                    
                    # 1. Structural Link (is_a)
                    parent = row.get('parent_category')
                    if parent:
                        parent = parent.strip()
                        self.add_relation(node, parent)
                        print (f"weapons_net.add_relation('{node}', '{parent}')")
                        is_a_count += 1

                    # 2. Property Links (has_a)
                    for key, value in row.items():
                        # Skip the structural columns
                        if key in ['node_name', 'parent_category']:
                            continue
                        
                        # Clean the value to handle numbers or missing data safely
                        val_str = str(value).strip() if value else ""
                        
                        # Skip if empty, 'na', or 'inherited'
                        if not val_str or val_str.lower() in ['na', 'inherited']:
                            continue
                        
                        # Handle lists like "[one, two]"
                        if val_str.startswith('[') and val_str.endswith(']'):
                            # Strip the brackets and split by comma
                            items = val_str[1:-1].split(',')
                            for item in items:
                                clean_item = item.strip()
                                if clean_item:
                                    self.add_property(node, key, clean_item)
                                    print (f"weapons_net.add_property('{node}', '{key}', '{clean_item}')")
                                    has_a_count += 1
                        else:
                            # Single value property
                            self.add_property(node, key, val_str)
                            print (f"weapons_net.add_property('{node}', '{key}', '{val_str}')")
                            has_a_count += 1
                            
            print_output("Imported network from CSV file", filename)
            print(f" - Added {is_a_count} IS_A facts")
            print(f" - Added {has_a_count} HAS_A facts")
            
        except FileNotFoundError:
            print_output("Error", f"File '{filename}' not found.", colour='red')
        except Exception as e:
            print_output("Failed", f"An error occurred: {e}", colour='red')


    # Create hierarchical visualisation
    def draw_hierarchical(self, filename, show_levels=False, **kwargs):

        # Handle Additional Arguments
        palette = kwargs.get('palette', self.palette)

        # Initiate the PyVis Network
        net = Network(height="1000px", width="100%", bgcolor=palette['bg'], font_color=palette['text'], directed=True)
        
        # We only want to draw nodes that are part of the structural 'is_a' tree.
        entity_nodes = set()
        for source, target, data in self.G.edges(data=True):
            if data.get('label') == 'is_a':
                entity_nodes.add(source)
                entity_nodes.add(target)

        # Sort the nodes by their depth level, then alphabetically. 
        sorted_nodes = sorted(list(entity_nodes), key=lambda x: (self.get_node_level(x) or 99, x))

        # Add Nodes with Property Lists (like a UML diagram)
        for node in sorted_nodes:
            props = []
            for _, target, data in self.G.out_edges(node, data=True):
                label = data.get('label')
                if label != 'is_a':
                    props.append(f"{label}: {target}")
            
            if props:
                props.sort()
                prop_text = "\n".join(props)
                display_label = f"{node.upper()}\n{'—'*len(node)}\n{prop_text}"
            else:
                display_label = f"{node.upper()}"

            # Calculate the nodes level
            level = self.get_node_level(node)
            if level is None: level = 99
            if node == self.root: level = 0
            
            color = palette.get(level, "#D3D3D3")

            # Are we showing the levels in the label?
            if show_levels:
                node_label = f'[{level}] {display_label}'
            else:
                node_label = f'{display_label}'

            net.add_node(
                node, 
                label=node_label,
                shape="box",
                color=color,
                level=level, 
                font={'face': 'Arial', 'align': 'left', 'color': palette['text'], 'size': 20}
            )

        # 3. Add ONLY the structural edges
        for source, target, data in self.G.edges(data=True):
            if data.get('label') == 'is_a':
                # In PyVis, to draw a tree top-down, edges usually point from Parent to Child.
                # If your network is Child -> Parent (e.g., long_sword -> sword), 
                # PyVis might draw it bottom-up unless we tell it otherwise in the layout options.
                net.add_edge(source, target, width=3, color=palette['edge_struct'])

        # 4. Define Hierarchical Options
        # 4. Define Strict Hierarchical Options
        options = {
            "layout": {
                "hierarchical": {
                    "enabled": True,
                    "direction": "UD",            # Up-Down (Level 0 at the top)
                    "sortMethod": "directed",
                    "shakeTowards": "roots",      # Pulls loose branches tightly up to their parents
                    "levelSeparation": 350,       
                    "nodeSpacing": 400,           
                    "treeSpacing": 350,           
                    "parentCentralization": True, 
                    "blockShifting": True,
                    "edgeMinimization": True      # Now works perfectly because levels are explicit
                }
            },
            "physics": {
                "enabled": False
            },
            "edges": {
                "smooth": {
                    "type": "cubicBezier",        
                    "forceDirection": "vertical", 
                    "roundness": 0.4
                },
                "arrows": {
                    "to": {"enabled": True}
                }
            }
        }
        
         # Convert options for the visualisation JavaScript
        net.set_options(json.dumps(options))

        # Write out the diagram to the specified file
        net.write_html(filename)
        print_output("Wrote File",f"Network diagram written to {filename}.")     


# Create a dynamic visualisation of the network using PyVis, saved as HTML file
    def draw_dynamic(self, filename, show_levels=False, **kwargs):

        # Handle Additional Arguments
        palette = kwargs.get('palette', self.palette)


        # Initiate the PyVis Network
        net = Network(height="1000", width="100%", bgcolor=palette['bg'], font_color=palette['text'], directed=True)
        
        # Add Nodes 
        for node in self.G.nodes():
            #level = dist.get(node, 99)
            level = self.get_node_level(node) or 99
            if node == self.root:
                level = 0
            
            color = palette.get(level, "#D3D3D3")

            # Dynamic Scaling of nodes and fonts
            if level==0:
                dynamic_node_size = 120
                dynamic_font_size = 120
            else:
                dynamic_font_size = max(80 - (level * 8), 25) if level != 99 else 20
                dynamic_node_size = max(70 - (level * 7), 15) if level != 99 else 10

            if show_levels:
                label = f'{node} (level {level})'
            else:
                label = node
            
            net.add_node(
                node, 
                label=label,
                color=color, 
                size = dynamic_node_size,
                font={'size': dynamic_font_size, 'face': 'arial', 'color': palette['text']}
            )

        # Add Edges
        for source, target, data in self.G.edges(data=True):
            label = data.get('label')

            # Check if it is a structural label, as in an 'is_a' type relationship
            is_structural = label == 'is_a'
        
            net.add_edge(
                    source, 
                    target, 
                    label=label,
                    width=5 if is_structural else 2,
                    color= palette['edge_struct'] if is_structural else palette['edge_prop'],
                    font={'color': palette['edge_label'], 'size': 20, 'strokeWidth': 0}
            )

        # Define options for network
        options = {
            "nodes": {
                "font": {
                    "size": 30 
                }
            },
            "edges": {
                "font": {
                    "size": 18,
                    "strokeWidth": 0
                },
                "color": {"inherit": False}
            },
            "physics": {
                "enabled": True,
                "barnesHut": {
                    "gravitationalConstant": -15000, # Push nodes apart
                    "springLength": 400 # Lengthen the edges
                },
                "solver": "barnesHut"
            },
            "interaction": {
                "hideEdgesOnDrag": False,
                "hideEdgesOnZoom": False,
                "zoomView": True
            }
        }
        
        # Convert options for the visualisation JavaScript
        net.set_options(json.dumps(options))

        # Write out the diagram to the specified file
        net.write_html(filename)
        print_output("Wrote File",f"Network diagram written to {filename}.")




The network can be populated manually, for example:

```
# Create new network object (Level 0)
weapons_net = SemanticNetwork(root_node="weapon", title="Medieval Weapons")

# Add weapons to the hierarchy

# Bladed Hand Weapon Categories
weapons_net.add_relation("sword", "bladed_hand_weapon")

# Properties of a Generic Sword
# Read as follows: a sword has_a (has_part) pommel
weapons_net.add_property("sword", "has_part", "pommel")
weapons_net.add_property("sword", "has_part", "hilt")
weapons_net.add_property("sword", "has_part", "blade")
weapons_net.add_property("sword", "causes_damage", "cut")
weapons_net.add_property("sword", "primary_material", "steel" )
```

# Populate Network

## Semantic Network - Medieval Weapons 

In [95]:
# Create new network object (Level 0)

weapons_net = SemanticNetwork(root_node='weapon', title="Medieval Weapons Semantic Network")
weapons_net.add_property('weapon', 'era_used', '476,1500')

# Add weapons to the hierarchy

### Level 1 - Categories
weapons_net.add_relation('melee_weapon', 'weapon')
weapons_net.add_property('melee_weapon', 'effective_against', 'personnel')
weapons_net.add_property('melee_weapon', 'attack_rate', 'fast')
weapons_net.add_property('melee_weapon', 'movement_speed', 'fast')
weapons_net.add_property('melee_weapon', 'use_case', 'attack')
weapons_net.add_property('melee_weapon', 'ammunition_type', 'no_ammunition')
weapons_net.add_property('melee_weapon', 'is_throwable', 'not_throwable')
weapons_net.add_property('melee_weapon', 'people_reqd', 'one_person')

weapons_net.add_relation('ranged_weapon', 'weapon')
weapons_net.add_property('ranged_weapon', 'effective_against', 'personnel')
weapons_net.add_property('ranged_weapon', 'attack_rate', 'medium')
weapons_net.add_property('ranged_weapon', 'movement_speed', 'fast')
weapons_net.add_property('ranged_weapon', 'use_case', 'attack')
weapons_net.add_property('ranged_weapon', 'is_throwable', 'not_throwable')
weapons_net.add_property('ranged_weapon', 'people_reqd', 'one_person')

weapons_net.add_relation('siege_weapon', 'weapon')
weapons_net.add_property('siege_weapon', 'movement_speed', 'slow')
weapons_net.add_property('siege_weapon', 'use_case', 'attack')
weapons_net.add_property('siege_weapon', 'is_throwable', 'not_throwable')
weapons_net.add_property('siege_weapon', 'people_reqd', 'a_team')
weapons_net.add_property('siege_weapon', 'has_edges', 'no_edges')

### Level 2 - Categories
# Melee Weapon Categories
weapons_net.add_relation('bladed_hand_weapon', 'melee_weapon')
weapons_net.add_property('bladed_hand_weapon', 'effective_range', 'close_quarters')
weapons_net.add_property('bladed_hand_weapon', 'attack_type', 'cut')
weapons_net.add_property('bladed_hand_weapon', 'has_part', 'blade')

weapons_net.add_relation('blunt_hand_weapon', 'melee_weapon')
weapons_net.add_property('blunt_hand_weapon', 'effective_range', 'close_quarters')
weapons_net.add_property('blunt_hand_weapon', 'attack_type', 'bash')
weapons_net.add_property('blunt_hand_weapon', 'primary_material', 'wood')

weapons_net.add_relation('polearm', 'melee_weapon')
weapons_net.add_property('polearm', 'effective_range', 'short')
weapons_net.add_property('polearm', 'attack_type', 'thrust')

# Ranged Weapon Categories
weapons_net.add_relation('artillery', 'ranged_weapon')
weapons_net.add_property('artillery', 'effective_against', 'fortifications')
weapons_net.add_property('artillery', 'effective_range', 'long')
weapons_net.add_property('artillery', 'attack_rate', 'slow')
weapons_net.add_property('artillery', 'movement_speed', 'slow')
weapons_net.add_property('artillery', 'people_reqd', 'a_team')

weapons_net.add_relation('bow', 'ranged_weapon')
weapons_net.add_property('bow', 'effective_range', 'medium')

weapons_net.add_relation('francisca', 'ranged_weapon')
weapons_net.add_property('francisca', 'effective_range', 'short')
weapons_net.add_property('francisca', 'attack_rate', 'fast')
weapons_net.add_property('francisca', 'ammunition_type', 'no_ammunition')
weapons_net.add_property('francisca', 'is_throwable', 'throwable')
weapons_net.add_property('francisca', 'era_used', '500,800')
weapons_net.add_property('francisca', 'has_part', 'axe_head')
weapons_net.add_property('francisca', 'historical_origin', 'francs')
weapons_net.add_property('francisca', 'length_inches', '18,21')
weapons_net.add_property('francisca', 'primary_material', 'wood')
weapons_net.add_property('francisca', 'secondary_material', 'iron')

weapons_net.add_relation('gun', 'ranged_weapon')
weapons_net.add_property('gun', 'effective_range', 'long')
weapons_net.add_property('gun', 'attack_rate', 'slow')

weapons_net.add_relation('javelin', 'ranged_weapon')
weapons_net.add_property('javelin', 'effective_range', 'medium')
weapons_net.add_property('javelin', 'attack_rate', 'slow')
weapons_net.add_property('javelin', 'ammunition_type', 'no_ammunition')
weapons_net.add_property('javelin', 'is_throwable', 'throwable')
weapons_net.add_property('javelin', 'alternate_name', 'light_spear')
weapons_net.add_property('javelin', 'alternate_name', 'france')
weapons_net.add_property('javelin', 'attack_type', 'puncture')
weapons_net.add_property('javelin', 'has_part', 'shaft')
weapons_net.add_property('javelin', 'has_part', 'head')
weapons_net.add_property('javelin', 'historical_origin', 'vikings')
weapons_net.add_property('javelin', 'length_inches', '72,96')

# Siege Weapon Categories
weapons_net.add_relation('battering_ram', 'siege_weapon')
weapons_net.add_property('battering_ram', 'effective_against', 'fortifications')
weapons_net.add_property('battering_ram', 'effective_range', 'close_quarters')
weapons_net.add_property('battering_ram', 'attack_rate', 'medium')
weapons_net.add_property('battering_ram', 'movement_speed', 'medium')
weapons_net.add_property('battering_ram', 'ammunition_type', 'no_ammunition')

weapons_net.add_relation('biological_weapon', 'siege_weapon')
weapons_net.add_property('biological_weapon', 'effective_against', 'personnel')
weapons_net.add_property('biological_weapon', 'effective_range', 'long')
weapons_net.add_property('biological_weapon', 'attack_rate', 'medium')
weapons_net.add_property('biological_weapon', 'is_throwable', 'throwable')

weapons_net.add_relation('chemical_weapon', 'siege_weapon')
weapons_net.add_property('chemical_weapon', 'effective_against', 'personnel')
weapons_net.add_property('chemical_weapon', 'effective_range', 'long')
weapons_net.add_property('chemical_weapon', 'attack_rate', 'medium')
weapons_net.add_property('chemical_weapon', 'is_throwable', 'throwable')

weapons_net.add_relation('psychological_weapon', 'siege_weapon')
weapons_net.add_property('psychological_weapon', 'effective_against', 'personnel')
weapons_net.add_property('psychological_weapon', 'effective_range', 'long')
weapons_net.add_property('psychological_weapon', 'attack_rate', 'medium')
weapons_net.add_property('psychological_weapon', 'movement_speed', 'medium')
weapons_net.add_property('psychological_weapon', 'ammunition_type', 'no_ammunition')

weapons_net.add_relation('moveable_structure', 'siege_weapon')
weapons_net.add_property('moveable_structure', 'effective_against', 'fortifications')
weapons_net.add_property('moveable_structure', 'ammunition_type', 'no_ammunition')

### Level 3 - Types of Weapons
weapons_net.add_relation('dagger', 'bladed_hand_weapon')
weapons_net.add_property('dagger', 'effective_range', 'close_quarters')
weapons_net.add_property('dagger', 'is_throwable', 'throwable')
weapons_net.add_property('dagger', 'attack_type', 'puncture')
weapons_net.add_property('dagger', 'has_edges', 'two_edges')
weapons_net.add_property('dagger', 'has_part', 'blade')
weapons_net.add_property('dagger', 'primary_material', 'steel')
weapons_net.add_property('dagger', 'used_by', 'commoners')
weapons_net.add_property('dagger', 'used_by', 'knights')

weapons_net.add_relation('knife', 'bladed_hand_weapon')
weapons_net.add_property('knife', 'effective_range', 'close_quarters')
weapons_net.add_property('knife', 'is_throwable', 'throwable')
weapons_net.add_property('knife', 'has_edges', 'one_edge')
weapons_net.add_property('knife', 'has_part', 'blade')
weapons_net.add_property('knife', 'primary_material', 'steel')
weapons_net.add_property('knife', 'secondary_material', 'wood')

weapons_net.add_relation('sword', 'bladed_hand_weapon')
weapons_net.add_property('sword', 'has_edges', 'two_edges')
weapons_net.add_property('sword', 'has_part', 'blade')
weapons_net.add_property('sword', 'has_part', 'hilt')
weapons_net.add_property('sword', 'has_part', 'pommel')
weapons_net.add_property('sword', 'primary_material', 'steel')

weapons_net.add_relation('club', 'blunt_hand_weapon')
weapons_net.add_property('club', 'attack_type', 'bash')

weapons_net.add_relation('mace', 'blunt_hand_weapon')
weapons_net.add_property('mace', 'cost', 'cheap')
weapons_net.add_property('mace', 'length_inches', '24,48')
weapons_net.add_property('mace', 'used_by', 'foot_soldiers')
weapons_net.add_property('mace', 'used_by', 'cavalry')

weapons_net.add_relation('becs_de_corbin', 'polearm')
weapons_net.add_property('becs_de_corbin', 'era_used', '1300,1500')
weapons_net.add_property('becs_de_corbin', 'alternate_name', 'war_hammer')
weapons_net.add_property('becs_de_corbin', 'attack_type', 'puncture')

weapons_net.add_relation('danish_axe', 'polearm')
weapons_net.add_property('danish_axe', 'is_throwable', 'throwable')
weapons_net.add_property('danish_axe', 'era_used', '700,1400')
weapons_net.add_property('danish_axe', 'alternate_name', 'broad_axe')
weapons_net.add_property('danish_axe', 'alternate_name', 'dane_axe')
weapons_net.add_property('danish_axe', 'attack_type', 'chop')
weapons_net.add_property('danish_axe', 'has_edges', 'one_edge')
weapons_net.add_property('danish_axe', 'has_part', 'shaft')
weapons_net.add_property('danish_axe', 'has_part', 'axe_head')
weapons_net.add_property('danish_axe', 'historical_origin', 'vikings')
weapons_net.add_property('danish_axe', 'length_inches', '48,72')

weapons_net.add_relation('fauchard', 'polearm')
weapons_net.add_property('fauchard', 'length_inches', '72,84')

weapons_net.add_relation('glaive', 'polearm')
weapons_net.add_property('glaive', 'era_used', '1000,1400')
weapons_net.add_property('glaive', 'attack_type', 'thrust')
weapons_net.add_property('glaive', 'has_part', 'blade')
weapons_net.add_property('glaive', 'has_part', 'shaft')
weapons_net.add_property('glaive', 'length_inches', '72,84')
weapons_net.add_property('glaive', 'used_by', 'foot_soldiers')
weapons_net.add_property('glaive', 'used_by', 'cavalry')

weapons_net.add_relation('guisarme', 'polearm')
weapons_net.add_property('guisarme', 'era_used', '1000,1400')
weapons_net.add_property('guisarme', 'alternate_name', 'giserne')
weapons_net.add_property('guisarme', 'alternate_name', 'bisarme')
weapons_net.add_property('guisarme', 'attack_type', 'dismount')
weapons_net.add_property('guisarme', 'has_part', 'shaft')
weapons_net.add_property('guisarme', 'has_part', 'hook')
weapons_net.add_property('guisarme', 'used_by', 'foot_soldiers')

weapons_net.add_relation('halberd', 'polearm')
weapons_net.add_property('halberd', 'era_used', '1300,1500')
weapons_net.add_property('halberd', 'alternate_name', 'swiss_voulge')
weapons_net.add_property('halberd', 'alternate_name', 'hellembart')
weapons_net.add_property('halberd', 'attack_type', 'chop')
weapons_net.add_property('halberd', 'attack_type', 'thrust')
weapons_net.add_property('halberd', 'has_edges', 'one_edge')
weapons_net.add_property('halberd', 'has_part', 'shaft')
weapons_net.add_property('halberd', 'has_part', 'axe_head')
weapons_net.add_property('halberd', 'has_part', 'spike')
weapons_net.add_property('halberd', 'historical_origin', 'swiss')

weapons_net.add_relation('lance', 'polearm')
weapons_net.add_property('lance', 'effective_against', 'cavalry')
weapons_net.add_property('lance', 'alternate_name', 'cavalry_spear')
weapons_net.add_property('lance', 'attack_type', 'puncture')
weapons_net.add_property('lance', 'has_part', 'shaft')
weapons_net.add_property('lance', 'has_part', 'tip')
weapons_net.add_property('lance', 'has_part', 'vamplate')
weapons_net.add_property('lance', 'historical_origin', 'romans')
weapons_net.add_property('lance', 'length_inches', '118,157')
weapons_net.add_property('lance', 'used_by', 'knights')
weapons_net.add_property('lance', 'used_by', 'foot_soldiers')

weapons_net.add_relation('maul', 'polearm')
weapons_net.add_property('maul', 'era_used', '1300,1500')
weapons_net.add_property('maul', 'attack_type', 'bash')
weapons_net.add_property('maul', 'has_part', 'shaft')
weapons_net.add_property('maul', 'has_part', 'hammer_head')
weapons_net.add_property('maul', 'used_by', 'archers')

weapons_net.add_relation('modern_flanged_mace', 'polearm')

weapons_net.add_relation('pike', 'polearm')
weapons_net.add_property('pike', 'effective_against', 'cavalry')
weapons_net.add_property('pike', 'effective_range', 'medium')
weapons_net.add_property('pike', 'era_used', '500,1700')
weapons_net.add_property('pike', 'alternate_name', 'geldon')
weapons_net.add_property('pike', 'attack_type', 'thrust')
weapons_net.add_property('pike', 'length_inches', '120,240')
weapons_net.add_property('pike', 'primary_material', 'wood')
weapons_net.add_property('pike', 'secondary_material', 'iron')
weapons_net.add_property('pike', 'secondary_material', 'steel')

weapons_net.add_relation('quarterstaff', 'polearm')
weapons_net.add_property('quarterstaff', 'era_used', '500,1800')
weapons_net.add_property('quarterstaff', 'alternate_name', 'staff')
weapons_net.add_property('quarterstaff', 'attack_type', 'bash')
weapons_net.add_property('quarterstaff', 'cost', 'cheap')
weapons_net.add_property('quarterstaff', 'has_edges', 'no_edges')
weapons_net.add_property('quarterstaff', 'primary_material', 'wood')

weapons_net.add_relation('spear', 'polearm')
weapons_net.add_property('spear', 'is_throwable', 'throwable')
weapons_net.add_property('spear', 'era_used', '500,1400')
weapons_net.add_property('spear', 'attack_type', 'thrust')
weapons_net.add_property('spear', 'has_part', 'shaft')
weapons_net.add_property('spear', 'has_part', 'head')
weapons_net.add_property('spear', 'historical_origin', 'stone_age')
weapons_net.add_property('spear', 'length_inches', '72,96')
weapons_net.add_property('spear', 'primary_material', 'wood')
weapons_net.add_property('spear', 'secondary_material', 'obsidian')
weapons_net.add_property('spear', 'secondary_material', 'iron')
weapons_net.add_property('spear', 'secondary_material', 'bronze')


###Level 4 - Leaves
weapons_net.add_relation('flail', 'blunt_hand_weapon')

weapons_net.add_relation('horsemans_pick', 'blunt_hand_weapon')

# Maces
weapons_net.add_relation('flanged_mace', 'mace')
weapons_net.add_property('flanged_mace', 'era_used', '900,1300')
weapons_net.add_property('flanged_mace', 'alternate_name', 'bardoukion')
weapons_net.add_property('flanged_mace', 'attack_type', 'puncture')
weapons_net.add_property('flanged_mace', 'historical_origin', 'byzantine_empire')

weapons_net.add_relation('holy_water_sprinkler', 'mace')
weapons_net.add_property('holy_water_sprinkler', 'alternate_name', 'goupillon')
weapons_net.add_property('holy_water_sprinkler', 'used_by', 'foot_soldiers')

weapons_net.add_relation('plançon a picot', 'mace')
weapons_net.add_property('plançon a picot', 'era_used', '1300,1400')
weapons_net.add_property('plançon a picot', 'attack_type', 'puncture')
weapons_net.add_property('plançon a picot', 'historical_origin', 'milan')
weapons_net.add_property('plançon a picot', 'used_by', 'cavalieri')

# Clubs
weapons_net.add_relation('morning_star', 'club')
weapons_net.add_property('morning_star', 'era_used', '1300,1400')
weapons_net.add_property('morning_star', 'used_by', 'foot_soldiers')
weapons_net.add_property('morning_star', 'used_by', 'peasant_militia')

# Danish Axes
weapons_net.add_relation('pollaxe', 'danish_axe')
weapons_net.add_property('pollaxe', 'era_used', '1300,1500')

weapons_net.add_relation('bardiche', 'pollaxe')

weapons_net.add_relation('sparth', 'danish_axe')
weapons_net.add_property('sparth', 'era_used', '1200,1400')
weapons_net.add_property('sparth', 'alternate_name', 'pale_axe')
weapons_net.add_property('sparth', 'alternate_name', 'sparr')
weapons_net.add_property('sparth', 'has_part', 'broad_blade')
weapons_net.add_property('sparth', 'historical_origin', 'scotland')
weapons_net.add_property('sparth', 'historical_origin', 'ireland')

# Spears
weapons_net.add_relation('winged_spear', 'spear')
weapons_net.add_property('winged_spear', 'is_throwable', 'throwable')
weapons_net.add_property('winged_spear', 'alternate_name', 'barred_spear')
weapons_net.add_property('winged_spear', 'has_part', 'shaft')
weapons_net.add_property('winged_spear', 'has_part', 'head')
weapons_net.add_property('winged_spear', 'has_part', 'wings')
weapons_net.add_property('winged_spear', 'historical_origin', 'romans')
weapons_net.add_property('winged_spear', 'historical_origin', 'francs')
weapons_net.add_property('winged_spear', 'historical_origin', 'vikings')

weapons_net.add_relation('bohemian_ear_spoon', 'winged_spear')

weapons_net.add_relation('spetum', 'winged_spear')

weapons_net.add_relation('ranseur', 'corsque')

weapons_net.add_relation('corsque', 'winged_spear')
weapons_net.add_property('corsque', 'alternate_name', 'chauve_souris')
weapons_net.add_property('corsque', 'attack_type', 'thrust')
weapons_net.add_property('corsque', 'has_part', 'three_bladed_head')

weapons_net.add_relation('partisan', 'winged_spear')

weapons_net.add_relation('angon', 'spear')
weapons_net.add_property('angon', 'effective_range', 'medium')
weapons_net.add_property('angon', 'is_throwable', 'throwable')
weapons_net.add_property('angon', 'attack_type', 'thrown')
weapons_net.add_property('angon', 'historical_origin', 'anglo_saxons')

# Bows
weapons_net.add_relation('cross_bow', 'bow')
weapons_net.add_property('cross_bow', 'effective_range', 'medium')
weapons_net.add_property('cross_bow', 'ammunition_type', 'bolt')
weapons_net.add_property('cross_bow', 'era_used', '500,1500')
weapons_net.add_property('cross_bow', 'alternate_name', 'frankish_bow')
weapons_net.add_property('cross_bow', 'attack_type', 'puncture')

weapons_net.add_relation('arbalest', 'cross_bow')
weapons_net.add_property('arbalest', 'era_used', '1300,1500')

weapons_net.add_relation('long_bow', 'bow')
weapons_net.add_property('long_bow', 'effective_range', 'long')
weapons_net.add_property('long_bow', 'ammunition_type', 'arrow')
weapons_net.add_property('long_bow', 'era_used', '1200,1500')
weapons_net.add_property('long_bow', 'alternate_name', 'self_bow')
weapons_net.add_property('long_bow', 'attack_type', 'puncture')
weapons_net.add_property('long_bow', 'length_inches', '72')
weapons_net.add_property('long_bow', 'primary_material', 'wood')
weapons_net.add_property('long_bow', 'primary_material', 'yew_wood')

# Guns
weapons_net.add_relation('arquebus', 'gun')
weapons_net.add_property('arquebus', 'ammunition_type', 'shot')
weapons_net.add_property('arquebus', 'ammunition_type', 'bullets')
weapons_net.add_property('arquebus', 'era_used', '1400,1600')
weapons_net.add_property('arquebus', 'alternate_name', 'hook_tube')
weapons_net.add_property('arquebus', 'attack_type', 'impact')
weapons_net.add_property('arquebus', 'has_part', 'barrel')
weapons_net.add_property('arquebus', 'has_part', 'stock')

weapons_net.add_relation('musket', 'arquebus')

weapons_net.add_relation('cavalier', 'arquebus')

weapons_net.add_relation('hand_cannon', 'gun')
weapons_net.add_property('hand_cannon', 'effective_range', 'medium')
weapons_net.add_property('hand_cannon', 'ammunition_type', 'stone')
weapons_net.add_property('hand_cannon', 'ammunition_type', 'iron_ball')
weapons_net.add_property('hand_cannon', 'ammunition_type', 'arrow')
weapons_net.add_property('hand_cannon', 'era_used', '1100,1600')
weapons_net.add_property('hand_cannon', 'alternate_name', 'gonne')
weapons_net.add_property('hand_cannon', 'attack_type', 'impact')
weapons_net.add_property('hand_cannon', 'has_part', 'barrel')
weapons_net.add_property('hand_cannon', 'has_part', 'stock')
weapons_net.add_property('hand_cannon', 'historical_origin', 'china')
weapons_net.add_property('hand_cannon', 'primary_material', 'metal')
weapons_net.add_property('hand_cannon', 'secondary_material', 'wood')
weapons_net.add_property('hand_cannon', 'secondary_material', 'bamboo')

# Artillery
weapons_net.add_relation('cannon', 'artillery')
weapons_net.add_property('cannon', 'ammunition_type', 'steel_ball')
weapons_net.add_property('cannon', 'attack_type', 'impact')

weapons_net.add_relation('stone_thrower', 'artillery')
weapons_net.add_property('stone_thrower', 'ammunition_type', 'stone')
weapons_net.add_property('stone_thrower', 'alternate_name', 'pierrier')
weapons_net.add_property('stone_thrower', 'attack_type', 'impact')
weapons_net.add_property('stone_thrower', 'primary_material', 'wood')

# Swords
weapons_net.add_relation('arming_sword', 'sword')
weapons_net.add_property('arming_sword', 'era_used', '1000,1350')
weapons_net.add_property('arming_sword', 'has_edges', 'two_edges')
weapons_net.add_property('arming_sword', 'length_inches', '30,32')
weapons_net.add_property('arming_sword', 'used_by', 'knight')

weapons_net.add_relation('broad_sword', 'sword')
weapons_net.add_property('broad_sword', 'era_used', '1600,1800')
weapons_net.add_property('broad_sword', 'attack_type', 'thrust')
weapons_net.add_property('broad_sword', 'has_edges', 'two_edges')
weapons_net.add_property('broad_sword', 'has_part', 'basket_hilt')
weapons_net.add_property('broad_sword', 'length_inches', '30,32')

weapons_net.add_relation('falchion', 'sword')
weapons_net.add_property('falchion', 'era_used', '1000,1500')
weapons_net.add_property('falchion', 'has_edges', 'one_edge')
weapons_net.add_property('falchion', 'historical_origin', 'europe')
weapons_net.add_property('falchion', 'length_inches', '30,32')
weapons_net.add_property('falchion', 'used_by', 'mounted_knight')

weapons_net.add_relation('long_sword', 'sword')
weapons_net.add_property('long_sword', 'era_used', '1250,1550')
weapons_net.add_property('long_sword', 'alternate_name', 'bastard_sword')
weapons_net.add_property('long_sword', 'alternate_name', 'great_sword')
weapons_net.add_property('long_sword', 'alternate_name', 'hand_and_a_half_sword')
weapons_net.add_property('long_sword', 'attack_type', 'thrust')
weapons_net.add_property('long_sword', 'has_edges', 'two_edges')
weapons_net.add_property('long_sword', 'has_part', 'fullers_aka_blood_grooves')
weapons_net.add_property('long_sword', 'historical_origin', 'europe')
weapons_net.add_property('long_sword', 'length_inches', '40,48')
weapons_net.add_property('long_sword', 'used_by', 'knight')
weapons_net.add_property('long_sword', 'weight_kg', '1.2,2.4')

# Falchions
weapons_net.add_relation('cleaver_falchion', 'falchion')
weapons_net.add_relation('cusped_falchion', 'falchion')


# Daggers
weapons_net.add_relation('anelace', 'dagger')
weapons_net.add_property('anelace', 'length_inches', '20,30')

weapons_net.add_relation('poingnard', 'dagger')
weapons_net.add_property('poingnard', 'era_used', '1100,1800')
weapons_net.add_property('poingnard', 'historical_origin', 'france')

weapons_net.add_relation('rondel', 'dagger')
weapons_net.add_property('rondel', 'length_inches', '20')

weapons_net.add_relation('stiletto', 'dagger')
weapons_net.add_property('stiletto', 'era_used', '1100,1800')
weapons_net.add_property('stiletto', 'alternate_name', 'misericorde')
weapons_net.add_property('stiletto', 'attack_type', 'puncture')
weapons_net.add_property('stiletto', 'has_edges', 'no_edges')
weapons_net.add_property('stiletto', 'historical_origin', 'england')
weapons_net.add_property('stiletto', 'historical_origin', 'germany')

# Cannons
weapons_net.add_relation('bombard', 'cannon')
weapons_net.add_property('bombard', 'ammunition_type', 'stone_ball')
weapons_net.add_property('bombard', 'era_used', '1350,1500')

weapons_net.add_relation('petard', 'cannon')
weapons_net.add_property('petard', 'effective_against', 'walls')
weapons_net.add_property('petard', 'effective_against', 'gates')
weapons_net.add_property('petard', 'effective_against', 'tunnels')
weapons_net.add_property('petard', 'ammunition_type', 'iron_shot')
weapons_net.add_property('petard', 'era_used', '1500,1600')
weapons_net.add_property('petard', 'people_reqd', 'one_person')
weapons_net.add_property('petard', 'alternate_name', 'a_bomb')
weapons_net.add_property('petard', 'attack_type', 'explosion')
weapons_net.add_property('petard', 'historical_origin', 'france')

# Stone Throwers
weapons_net.add_relation('ballista', 'stone_thrower')
weapons_net.add_property('ballista', 'ammunition_type', 'iron_tipped_bolt')
weapons_net.add_property('ballista', 'era_used', '1200,1500')

weapons_net.add_relation('counterweight_trebuchet', 'stone_thrower')
weapons_net.add_property('counterweight_trebuchet', 'attack_rate', 'fast')
weapons_net.add_property('counterweight_trebuchet', 'era_used', '1100,1400')
weapons_net.add_property('counterweight_trebuchet', 'historical_origin', 'middle_east')

weapons_net.add_relation('mangonel', 'stone_thrower')
weapons_net.add_property('mangonel', 'ammunition_type', 'multiple_stones')
weapons_net.add_property('mangonel', 'ammunition_type', 'burning_pitch')
weapons_net.add_property('mangonel', 'ammunition_type', 'flammable_pots')
weapons_net.add_property('mangonel', 'era_used', '1200,1500')
weapons_net.add_property('mangonel', 'has_part', 'fixed_bowl')

weapons_net.add_relation('onager', 'stone_thrower')
weapons_net.add_property('onager', 'effective_range', 'medium')
weapons_net.add_property('onager', 'ammunition_type', 'stones')
weapons_net.add_property('onager', 'ammunition_type', 'burning_pitch')
weapons_net.add_property('onager', 'has_part', 'sling_basket')
weapons_net.add_property('onager', 'historical_origin', 'romans')

weapons_net.add_relation('springald', 'stone_thrower')
weapons_net.add_property('springald', 'effective_range', 'medium')
weapons_net.add_property('springald', 'ammunition_type', 'heavy_bolt')
weapons_net.add_property('springald', 'era_used', '1200,1500')

weapons_net.add_relation('traction_trebuchet', 'stone_thrower')
weapons_net.add_property('traction_trebuchet', 'era_used', '400,1400')
weapons_net.add_property('traction_trebuchet', 'people_reqd', '15_45')
weapons_net.add_property('traction_trebuchet', 'historical_origin', 'china')

# Siege Weapons
weapons_net.add_relation('cat_and_weasel', 'moveable_structure')
weapons_net.add_property('cat_and_weasel', 'effective_range', 'close_quarters')
weapons_net.add_property('cat_and_weasel', 'attack_rate', 'slow')
weapons_net.add_property('cat_and_weasel', 'movement_speed', 'medium')

weapons_net.add_relation('siege_tower', 'moveable_structure')
weapons_net.add_property('siege_tower', 'effective_range', 'short')
weapons_net.add_property('siege_tower', 'attack_rate', 'slow')


### Level 5 - Specific Instances / Famous Weapons
weapons_net.add_relation('medici_falchion', 'falchion')
weapons_net.add_property('medici_falchion', 'use_case', 'ceremonial')
weapons_net.add_property('medici_falchion', 'era_used', '1560')
weapons_net.add_property('medici_falchion', 'decoration', 'coat_of_arms_of_cosimo_de_medici')
weapons_net.add_property('medici_falchion', 'historical_origin', 'florence')
weapons_net.add_property('medici_falchion', 'secondary_material', 'gold_plated')

weapons_net.add_relation('the_wallace_sword', 'long_sword')
weapons_net.add_property('the_wallace_sword', 'era_used', '1298')
weapons_net.add_property('the_wallace_sword', 'historical_origin', 'scotland')
weapons_net.add_property('the_wallace_sword', 'used_by', 'william_wallace')
weapons_net.add_property('the_wallace_sword', 'weight_kg', '2.7')

weapons_net.add_relation('mace_of_bishop_odo', 'mace')
weapons_net.add_property('mace_of_bishop_odo', 'era_used', '1066')
weapons_net.add_property('mace_of_bishop_odo', 'used_by', 'bishop_odo_of_bayeux')

weapons_net.add_relation('almace', 'arming_sword')
weapons_net.add_property('almace', 'era_used', '778')
weapons_net.add_property('almace', 'used_by', 'archbishop_turpin')

weapons_net.add_relation('royal_armouries_holy_water_sprinkler', 'holy_water_sprinkler')
weapons_net.add_property('royal_armouries_holy_water_sprinkler', 'use_case', 'ceremonial')
weapons_net.add_property('royal_armouries_holy_water_sprinkler', 'length_inches', '74.5')

weapons_net.add_relation('wallace_collection_morning_star', 'morning_star')
weapons_net.add_property('wallace_collection_morning_star', 'use_case', 'ceremonial')
weapons_net.add_property('wallace_collection_morning_star', 'era_used', '1500,1600')
weapons_net.add_property('wallace_collection_morning_star', 'primary_material', 'steel')
weapons_net.add_property('wallace_collection_morning_star', 'secondary_material', 'gold,silver')


The network can also be created from existing Prolog or JSON files

In [96]:
# Import from CSV file
# Commented out as defining network manually above
# weapons_net.import_from_csv('0-Medieval_Weapons_Source.csv')

# Export to JSON
weapons_net.export_to_json(filename='2-Medieval_Weapons_Network.json')

# Export to CSV
weapons_net.export_to_csv(filename='2-Medieval_Weapons_Network.csv')

# Export to Prolog
weapons_net.export_to_prolog(filename='3-Medieval_Weapons_Network.pl', sort=True)

on_screen_palette = {'bg':"#000000", 
                     'text': "#ffffff", 
                     'edge_label': "#F0D90A", 
                     'edge_struct': "#c4830b81", 
                     'edge_prop': "#7847ff81",
                     0:"#C3ADFF", 
                     1:"#9670ff", 
                     2:"#7847ff", 
                     3:"#5A1FFF", 
                     4:"#4100F5", 
                     5:"#3100B8", 
                     6:"#22007C"}

print_palette =     {'bg':"#ffffff", 
                     'text': "#000000", 
                     'edge_label': "#111111", 
                     'edge_struct': "#333333", 
                     'edge_prop': "#666666",
                     0:"#dddddd", 
                     1:"#bbbbbb", 
                     2:"#aaaaaa", 
                     3:"#999999", 
                     4:"#888888", 
                     5:"#777777", 
                     6:"#666666"}

# Draw dynamic PyVis diagrams
weapons_net.draw_dynamic(filename='4-Medieval_Weapons_Network_Bubbles.html',show_levels=True, palette=on_screen_palette)
weapons_net.draw_hierarchical(filename='4-Medieval_Weapons_Network.html',show_levels=True, palette=on_screen_palette)
weapons_net.draw_dynamic(filename='4-Medieval_Weapons_Network_Bubbles_Printable.html',show_levels=True, palette=print_palette)
weapons_net.draw_hierarchical(filename='4-Medieval_Weapons_Network_Printable.html',show_levels=True, palette=print_palette)

Successfully saved: 262 nodes to 2-Medieval_Weapons_Network.json
Wrote File: Exported 85 entity rows to 2-Medieval_Weapons_Network.csv
Wrote File: Prolog written to 3-Medieval_Weapons_Network.pl.
Wrote File: Network diagram written to 4-Medieval_Weapons_Network_Bubbles.html.
Wrote File: Network diagram written to 4-Medieval_Weapons_Network.html.
Wrote File: Network diagram written to 4-Medieval_Weapons_Network_Bubbles_Printable.html.
Wrote File: Network diagram written to 4-Medieval_Weapons_Network_Printable.html.


In [97]:
sys.exit('Stopping notebook run here.')  

SystemExit: Stopping notebook run here.

In [ ]:
# Create network
weapons_net = SemanticNetwork (root_node='weapon', title="Medieval Weapons")

# Import JSON File
weapons_net.import_from_json(filename='2-Medieval_Weapons_Network.json')

# Import Prolog File
# weapons_net.import_from_prolog(filename='3-Medieval_Weapons_Network.pl')


Successfully loaded: 262 nodes from 2-Medieval_Weapons_Network.json
